In [5]:
# -*- coding: utf-8 -*-
"""
CREACIÓN DE BASE H3 (SIN FILTRO DE CRÉDITO FORMAL 2024)
=============================================================================
Versión corregida V2: Maneja correctamente CreditoInformal_2025
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURACIÓN DE RUTAS
# ============================================================

ruta_base = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25_v2"
ruta_panel = os.path.join(ruta_base, "PANEL_2024_2025.csv")
ruta_salida = os.path.join(ruta_base, "BASE_H3.csv")

print("="*70)
print("CREANDO BASE ESPECÍFICA PARA H3 (V2)")
print("="*70)
print(f"\n📂 Panel fuente: {ruta_panel}")
print(f"📂 Salida: {ruta_salida}")

# ============================================================
# 2. CARGAR PANEL
# ============================================================

print("\n📊 Cargando panel...")
df = pd.read_csv(ruta_panel, encoding="utf-8-sig", low_memory=False)
print(f"✅ Panel cargado: {len(df):,} filas, {len(df.columns)} columnas")

# Verificar qué columnas de crédito existen
print("\n🔍 Columnas relacionadas con crédito en el panel:")
for col in df.columns:
    if 'credito' in col.lower() or 'Credito' in col:
        print(f"  • {col}")

# ============================================================
# 3. CONVERTIR VARIABLES A NUMÉRICO
# ============================================================

print("\n🔄 Convirtiendo variables a numérico...")

df["P203_num"] = pd.to_numeric(df["P203"], errors="coerce")
df["Ocupado_num"] = pd.to_numeric(df["Ocupado"], errors="coerce")
df["CreditoFormal_2024_num"] = pd.to_numeric(df["CreditoFormal_2024"], errors="coerce")
df["CreditoFormal_2025_num"] = pd.to_numeric(df["CreditoFormal_2025"], errors="coerce")
df["Informal_num"] = pd.to_numeric(df["Informal"], errors="coerce")
df["P301A_num"] = pd.to_numeric(df["P301A"], errors="coerce")
df["TenenciaBilletera_num"] = pd.to_numeric(df["TenenciaBilletera"], errors="coerce")
df["UsoBilletera_num"] = pd.to_numeric(df["UsoBilletera"], errors="coerce")

# ⚠️ CORREGIDO: La variable en el panel se llama "CreditoInformal_2025"
if "CreditoInformal_2025" in df.columns:
    df["CreditoInformal_num"] = pd.to_numeric(df["CreditoInformal_2025"], errors="coerce")
    print("  ✓ CreditoInformal_2025 encontrada y convertida a numérico")
else:
    df["CreditoInformal_num"] = pd.NA
    print("  ⚠️ CreditoInformal_2025 NO encontrada en el panel")

print("✅ Conversión completada")

# ============================================================
# 4. APLICAR FILTROS (EXCLUYENDO EL DE CRÉDITO FORMAL 2024)
# ============================================================

print("\n🔍 Aplicando filtros...")

n_inicial = len(df)
print(f"  • Inicial: {n_inicial:,}")

# Filtro 1: Excluir P301A == 99
df = df[df["P301A_num"] != 99]
print(f"  • Excluir P301A==99: {len(df):,} ({len(df)/n_inicial*100:.1f}%)")

# Filtro 2: Solo jefes de hogar
n_antes = len(df)
df = df[df["P203_num"] == 1]
print(f"  • Solo jefes de hogar: {len(df):,} ({len(df)/n_antes*100:.1f}%)")

# Filtro 3: Solo ocupados
n_antes = len(df)
df = df[df["Ocupado_num"] == 1]
print(f"  • Solo ocupados: {len(df):,} ({len(df)/n_antes*100:.1f}%)")

# ⚠️ FILTRO ELIMINADO: NO se filtra por CreditoFormal_2024 == 0
print(f"  • 🔴 FILTRO ELIMINADO: NO se restringe por crédito formal 2024")
print(f"  • Se mantienen TODOS los trabajadores (con y sin crédito formal)")

# Filtro 4: Sin missing en variables clave
n_antes = len(df)
variables_clave = ["Informal_num", "TenenciaBilletera_num", "UsoBilletera_num", 
                   "P207", "Edad", "P301A_num"]
df = df.dropna(subset=variables_clave)
print(f"  • Sin missing en variables clave: {len(df):,} ({len(df)/n_antes*100:.1f}%)")

# ============================================================
# 5. CREAR VARIABLE NUEVO CRÉDITO FORMAL
# ============================================================

df["NuevoCredito"] = (df["CreditoFormal_2025_num"] == 1).astype(int)

# ============================================================
# 6. RENOMBRAR COLUMNAS (CORREGIDO)
# ============================================================

print("\n📝 Renombrando columnas...")

diccionario = {
    "llave_persona": "id_persona",
    "P203": "jefe_hogar",
    "P207": "sexo",
    "Edad": "edad",
    "P301A": "nivel_educativo",
    "Ocupado": "ocupado",
    "Informal": "trabajador_informal",
    "TenenciaBilletera": "tiene_billetera",
    "UsoBilletera": "usa_billetera",
    "CreditoFormal_2024": "credito_formal_2024",
    "CreditoFormal_2025": "credito_formal_2025",
    "CreditoInformal_2025": "credito_informal_2025",  # ← CORREGIDO: mapeo correcto
    "NuevoCredito": "nuevo_credito_formal",
    "FACTOR07": "factor_expansion",
    "ESTRATO": "estrato",
    "DOMINIO": "dominio",
    "MIEPERHO": "miembros_hogar",
    "CONGLOME": "conglomerado",
    "ingreso_percapita_2024": "ingreso_percapita_2024",
    "gasto_percapita_2024": "gasto_percapita_2024",
    "ingreso_percapita_2025": "ingreso_percapita_2025",
    "gasto_percapita_2025": "gasto_percapita_2025",
    "RazonNoCredito": "razon_no_credito",
    "Razon_Demanda": "razon_demanda",
    "Razon_Deuda": "razon_deuda",
    "Razon_Intereses": "razon_intereses",
    "Razon_Costos": "razon_costos",
    "Razon_Requisitos": "razon_requisitos",
    "Razon_Infocorp": "razon_infocorp",
    "Razon_Otro": "razon_otro",
    "tiene_barrera_credito": "tiene_barrera_credito",
}

# Renombrar solo las columnas que existen
columnas_a_renombrar = {k: v for k, v in diccionario.items() if k in df.columns}
df = df.rename(columns=columnas_a_renombrar)

# ============================================================
# 7. ASEGURAR QUE credito_informal_2025 SEA NUMÉRICA
# ============================================================

# Si la columna renombrada existe pero es objeto, convertir a numérico
if 'credito_informal_2025' in df.columns:
    df['credito_informal_2025'] = pd.to_numeric(df['credito_informal_2025'], errors='coerce').fillna(0)
    print("  ✓ credito_informal_2025 convertida a numérico")
else:
    # Si no existe, crearla desde CreditoInformal_num
    if 'CreditoInformal_num' in df.columns:
        df['credito_informal_2025'] = pd.to_numeric(df['CreditoInformal_num'], errors='coerce').fillna(0)
        print("  ✓ credito_informal_2025 creada desde CreditoInformal_num")
    else:
        df['credito_informal_2025'] = 0
        print("  ⚠️ credito_informal_2025 creada como columna de ceros (default)")

# ============================================================
# 8. ELIMINAR COLUMNAS AUXILIARES
# ============================================================

columnas_auxiliares = ["P203_num", "Ocupado_num", "CreditoFormal_2024_num", 
                       "CreditoFormal_2025_num", "Informal_num", "P301A_num",
                       "TenenciaBilletera_num", "UsoBilletera_num"]

if "CreditoInformal_num" in df.columns:
    columnas_auxiliares.append("CreditoInformal_num")

# También eliminar la columna original CreditoInformal_2025 si quedó duplicada
if "CreditoInformal_2025" in df.columns and "credito_informal_2025" in df.columns:
    df = df.drop(columns=["CreditoInformal_2025"])

df = df.drop(columns=[c for c in columnas_auxiliares if c in df.columns])

# ============================================================
# 9. GUARDAR BASE
# ============================================================

print("\n💾 Guardando base...")
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
print(f"✅ BASE_H3.csv guardado exitosamente")
print(f"   • {len(df):,} filas")
print(f"   • {len(df.columns):,} columnas")

# ============================================================
# 10. VERIFICACIÓN Y ESTADÍSTICAS (CORREGIDA)
# ============================================================

print("\n" + "="*70)
print("VERIFICACIÓN DE BASE_H3")
print("="*70)

print(f"\n📊 Estadísticas descriptivas:")
print(f"  • Total observaciones: {len(df):,}")

# trabajador_informal
if 'trabajador_informal' in df.columns:
    print(f"  • trabajador_informal: {df['trabajador_informal'].sum():,} ({df['trabajador_informal'].mean():.2%})")

# credito_formal_2024
if 'credito_formal_2024' in df.columns:
    print(f"  • credito_formal_2024: {df['credito_formal_2024'].sum():,} ({df['credito_formal_2024'].mean():.2%})")

# credito_informal_2025 - AHORA NUMÉRICA
if 'credito_informal_2025' in df.columns:
    print(f"  • credito_informal_2025: {df['credito_informal_2025'].sum():,} ({df['credito_informal_2025'].mean():.2%})")
    print(f"    - Tipo de dato: {df['credito_informal_2025'].dtype}")

# usa_billetera
if 'usa_billetera' in df.columns:
    print(f"  • usa_billetera: {df['usa_billetera'].sum():,} ({df['usa_billetera'].mean():.2%})")

# nuevo_credito_formal
if 'nuevo_credito_formal' in df.columns:
    print(f"  • nuevo_credito_formal: {df['nuevo_credito_formal'].sum():,} ({df['nuevo_credito_formal'].mean():.2%})")

# ============================================================
# 11. COMPARACIÓN CON MUESTRA RESTRINGIDA (si existe)
# ============================================================

ruta_restringida = os.path.join(ruta_base, "BASE_REGRESIONES.csv")
if os.path.exists(ruta_restringida):
    df_rest = pd.read_csv(ruta_restringida, encoding="utf-8-sig", low_memory=False)
    
    print("\n📊 COMPARACIÓN CON MUESTRA RESTRINGIDA (BASE_REGRESIONES):")
    print("-"*70)
    print(f"  {'Métrica':<30} {'BASE_REGRESIONES':>20} {'BASE_H3':>15} {'Diferencia':>15}")
    print("-"*70)
    
    # Total
    n_rest = len(df_rest)
    n_h3 = len(df)
    print(f"  {'Total N':<30} {n_rest:>20,} {n_h3:>15,} {n_h3 - n_rest:>+15,}")
    
    # Informales
    if 'trabajador_informal' in df_rest.columns and 'trabajador_informal' in df.columns:
        inf_rest = df_rest['trabajador_informal'].sum()
        inf_h3 = df['trabajador_informal'].sum()
        print(f"  {'Informales (N)':<30} {inf_rest:>20,} {inf_h3:>15,} {inf_h3 - inf_rest:>+15,}")
    
    # Con crédito formal 2024
    if 'credito_formal_2024' in df_rest.columns and 'credito_formal_2024' in df.columns:
        cf_rest = df_rest['credito_formal_2024'].sum()
        cf_h3 = df['credito_formal_2024'].sum()
        print(f"  {'Con crédito formal 2024':<30} {cf_rest:>20,} {cf_h3:>15,} {cf_h3 - cf_rest:>+15,}")
    
    # Usuarios de billetera (informales) - CORREGIDO: usar sum() sin .values
    if 'trabajador_informal' in df_rest.columns and 'usa_billetera' in df_rest.columns:
        try:
            billetera_rest = df_rest[df_rest['trabajador_informal'] == 1]['usa_billetera'].sum()
            billetera_h3 = df[df['trabajador_informal'] == 1]['usa_billetera'].sum()
            print(f"  {'Billetera (informales)':<30} {billetera_rest:>20,} {billetera_h3:>15,} {billetera_h3 - billetera_rest:>+15,}")
        except:
            print(f"  {'Billetera (informales)':<30} {'N/A':>20} {'N/A':>15} {'N/A':>15}")
    
    print("-"*70)
    print(f"  ➡️ Ganancia de observaciones para H3: {n_h3 - n_rest:+,} filas")

# ============================================================
# 12. LISTA DE VARIABLES FINALES
# ============================================================

print(f"\n📋 Lista de variables en BASE_H3 ({len(df.columns)} columnas):")
print("-"*70)
for i, col in enumerate(df.columns, 1):
    # Marcar si es numérica o no
    tipo = "num" if pd.api.types.is_numeric_dtype(df[col]) else "obj"
    # Mostrar algunos valores para verificar
    if col == 'credito_informal_2025':
        print(f"  {i:>2}. {col} ({tipo}) - valores: {df[col].unique()[:5]}")
    else:
        print(f"  {i:>2}. {col} ({tipo})")

print("\n" + "="*70)
print("✅ PROCESO COMPLETADO EXITOSAMENTE")
print("="*70)

CREANDO BASE ESPECÍFICA PARA H3 (V2)

📂 Panel fuente: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25_v2\PANEL_2024_2025.csv
📂 Salida: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25_v2\BASE_H3.csv

📊 Cargando panel...
✅ Panel cargado: 32,079 filas, 30 columnas

🔍 Columnas relacionadas con crédito en el panel:
  • CreditoFormal_2024
  • CreditoFormal_2025
  • RazonNoCredito
  • tiene_barrera_credito
  • CreditoInformal_2025

🔄 Convirtiendo variables a numérico...
  ✓ CreditoInformal_2025 encontrada y convertida a numérico
✅ Conversión completada

🔍 Aplicando filtros...
  • Inicial: 32,079
  • Excluir P301A==99: 32,052 (99.9%)
  • Solo jefes de hogar: 9,090 (28.4%)
  • Solo ocupados: 7,708 (84.8%)
  • 🔴 FILTRO ELIMINADO: NO se restringe por crédito formal 2024
  • Se mantienen TODOS los trabajadores (con y sin crédito formal)
  • Sin missing en variables clave: 7,708 (100.0%)

📝 Renombrando columnas...
  ✓ credito_informal_

In [6]:
# -*- coding: utf-8 -*-
"""
VERIFICACIÓN COMPLETA DE BASE_H3.csv
=============================================================================
Este script revisa TODAS las variables de la base para asegurar que:
1. Los tipos de datos son correctos
2. No hay valores missing problemáticos
3. Los rangos son válidos
4. Las variables categóricas tienen las categorías esperadas
5. Las correlaciones tienen sentido
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CARGAR BASE
# ============================================================

ruta_base = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25_v2"
ruta_h3 = os.path.join(ruta_base, "BASE_H3.csv")

print("="*70)
print("VERIFICACIÓN COMPLETA DE BASE_H3.csv")
print("="*70)

# Cargar base
df = pd.read_csv(ruta_h3, encoding="utf-8-sig", low_memory=False)
print(f"\n✅ Base cargada: {len(df):,} filas, {len(df.columns)} columnas")

# ============================================================
# 2. VERIFICAR ESTRUCTURA GENERAL
# ============================================================

print("\n" + "="*70)
print("1. ESTRUCTURA GENERAL")
print("="*70)

print(f"\n📊 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")

# Memoria usada
memoria = df.memory_usage(deep=True).sum() / 1024**2
print(f"💾 Memoria usada: {memoria:.2f} MB")

# ============================================================
# 3. VERIFICAR TIPOS DE DATOS
# ============================================================

print("\n" + "="*70)
print("2. TIPOS DE DATOS POR VARIABLE")
print("="*70)

print("\n📋 Tipos de datos:")
print("-"*70)
print(f"{'Variable':<35} {'Tipo actual':>15} {'Tipo esperado':>15} {'¿OK?':>10}")
print("-"*70)

# Diccionario de tipos esperados
tipos_esperados = {
    'id_persona': 'object',
    'jefe_hogar': 'int64',
    'sexo': 'object',
    'edad': 'int64',
    'ocupado': 'int64',
    'trabajador_informal': 'int64',
    'tiene_billetera': 'int64',
    'usa_billetera': 'int64',
    'credito_formal_2024': 'int64',
    'factor_expansion': 'float64',
    'nivel_educativo': 'int64',
    'estrato': 'int64',
    'dominio': 'int64',
    'miembros_hogar': 'int64',
    'conglomerado': 'int64',
    'ingreso_percapita_2024': 'float64',
    'gasto_percapita_2024': 'float64',
    'credito_formal_2025': 'int64',
    'ingreso_percapita_2025': 'float64',
    'gasto_percapita_2025': 'float64',
    'razon_no_credito': 'float64',
    'razon_demanda': 'int64',
    'razon_deuda': 'int64',
    'razon_intereses': 'int64',
    'razon_costos': 'int64',
    'razon_requisitos': 'int64',
    'razon_infocorp': 'int64',
    'razon_otro': 'int64',
    'tiene_barrera_credito': 'int64',
    'credito_informal_2025': 'int64',
    'nuevo_credito_formal': 'int64'
}

ok_total = 0
for col in df.columns:
    tipo_actual = str(df[col].dtype)
    tipo_esperado = tipos_esperados.get(col, 'No especificado')
    
    # Verificar si el tipo es compatible (permite int64, float64, object)
    es_ok = False
    if tipo_esperado == 'object' and tipo_actual == 'object':
        es_ok = True
    elif 'int' in tipo_actual and 'int' in tipo_esperado:
        es_ok = True
    elif 'float' in tipo_actual and ('float' in tipo_esperado or 'int' in tipo_esperado):
        es_ok = True
    elif tipo_actual == tipo_esperado:
        es_ok = True
    
    ok_total += 1 if es_ok else 0
    print(f"{col:<35} {tipo_actual:>15} {tipo_esperado:>15} {'✅' if es_ok else '❌':>10}")

print("-"*70)
print(f"✅ {ok_total}/{len(df.columns)} variables con tipo correcto")

# ============================================================
# 4. VERIFICAR VALORES MISSING
# ============================================================

print("\n" + "="*70)
print("3. VALORES MISSING")
print("="*70)

print(f"\n📊 Resumen de valores missing:")
print("-"*70)
print(f"{'Variable':<35} {'Missing':>12} {'% Missing':>12} {'¿OK?':>10}")
print("-"*70)

missing_ok = 0
for col in df.columns:
    n_missing = df[col].isna().sum()
    pct_missing = n_missing / len(df) * 100
    es_ok = n_missing == 0
    missing_ok += 1 if es_ok else 0
    print(f"{col:<35} {n_missing:>12,} {pct_missing:>11.2f}% {'✅' if es_ok else '⚠️':>10}")

print("-"*70)
print(f"✅ {missing_ok}/{len(df.columns)} variables sin missing")

# Mostrar variables con missing (si hay)
if missing_ok < len(df.columns):
    print("\n⚠️ Variables con valores missing:")
    for col in df.columns:
        n_missing = df[col].isna().sum()
        if n_missing > 0:
            print(f"  - {col}: {n_missing:,} ({n_missing/len(df)*100:.1f}%)")

# ============================================================
# 5. VERIFICAR ESTADÍSTICAS DESCRIPTIVAS
# ============================================================

print("\n" + "="*70)
print("4. ESTADÍSTICAS DESCRIPTIVAS")
print("="*70)

# Seleccionar variables numéricas
variables_numericas = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"\n📊 Estadísticas para variables numéricas ({len(variables_numericas)} variables):")
print("-"*70)

for col in variables_numericas:
    # Estadísticas básicas
    media = df[col].mean()
    mediana = df[col].median()
    min_val = df[col].min()
    max_val = df[col].max()
    std_val = df[col].std()
    n_valid = df[col].count()
    
    print(f"\n📌 {col}:")
    print(f"  • N válidos: {n_valid:,}")
    print(f"  • Media: {media:.4f}")
    print(f"  • Mediana: {mediana:.4f}")
    print(f"  • Mínimo: {min_val:.4f}")
    print(f"  • Máximo: {max_val:.4f}")
    print(f"  • Desv. estándar: {std_val:.4f}")
    
    # Verificar rangos esperados para variables binarias
    if col in ['jefe_hogar', 'ocupado', 'trabajador_informal', 'tiene_billetera', 
               'usa_billetera', 'credito_formal_2024', 'credito_formal_2025',
               'credito_informal_2025', 'nuevo_credito_formal', 'razon_demanda',
               'razon_deuda', 'razon_intereses', 'razon_costos', 'razon_requisitos',
               'razon_infocorp', 'razon_otro', 'tiene_barrera_credito']:
        valores_unicos = df[col].unique()
        if set(valores_unicos).issubset({0, 1, np.nan}):
            print(f"  • ✅ Variable binaria (0/1) - valores: {sorted([x for x in valores_unicos if not pd.isna(x)])}")
        else:
            print(f"  • ⚠️ Variable binaria pero tiene valores extraños: {sorted(valores_unicos)}")

# ============================================================
# 6. VERIFICAR VARIABLES CATEGÓRICAS
# ============================================================

print("\n" + "="*70)
print("5. VARIABLES CATEGÓRICAS")
print("="*70)

# Variables categóricas esperadas
variables_categoricas = ['id_persona', 'sexo', 'dominio', 'estrato']

print("\n📊 Distribución de variables categóricas:")
print("-"*70)

for col in variables_categoricas:
    if col in df.columns:
        print(f"\n📌 {col}:")
        print(f"  • Valores únicos: {df[col].nunique()}")
        print(f"  • Top 5 valores:")
        for valor, count in df[col].value_counts().head(5).items():
            pct = count / len(df) * 100
            print(f"    - {valor}: {count:,} ({pct:.1f}%)")

# ============================================================
# 7. VERIFICAR VARIABLE DEPENDIENTE (CRÉDITO INFORMAL)
# ============================================================

print("\n" + "="*70)
print("6. VARIABLE DEPENDIENTE: credito_informal_2025")
print("="*70)

if 'credito_informal_2025' in df.columns:
    print(f"\n📊 Estadísticas de credito_informal_2025:")
    print(f"  • Total observaciones: {len(df):,}")
    print(f"  • Con crédito informal: {df['credito_informal_2025'].sum():,} ({df['credito_informal_2025'].mean():.2%})")
    print(f"  • Sin crédito informal: {(df['credito_informal_2025'] == 0).sum():,} ({(df['credito_informal_2025'] == 0).mean():.2%})")
    print(f"  • Tipo de dato: {df['credito_informal_2025'].dtype}")
    
    # Verificar si es binaria (0/1)
    valores = df['credito_informal_2025'].unique()
    print(f"  • Valores únicos: {sorted(valores)}")
    
    if set(valores).issubset({0, 1}):
        print("  • ✅ Es una variable binaria válida (0/1)")
    else:
        print("  • ❌ La variable NO es binaria (0/1)")

# ============================================================
# 8. VERIFICAR VARIABLE INDEPENDIENTE (BILLETERA)
# ============================================================

print("\n" + "="*70)
print("7. VARIABLE INDEPENDIENTE: usa_billetera")
print("="*70)

if 'usa_billetera' in df.columns:
    print(f"\n📊 Estadísticas de usa_billetera:")
    print(f"  • Total observaciones: {len(df):,}")
    print(f"  • Usan billetera: {df['usa_billetera'].sum():,} ({df['usa_billetera'].mean():.2%})")
    print(f"  • No usan billetera: {(df['usa_billetera'] == 0).sum():,} ({(df['usa_billetera'] == 0).mean():.2%})")
    print(f"  • Tipo de dato: {df['usa_billetera'].dtype}")
    
    valores = df['usa_billetera'].unique()
    print(f"  • Valores únicos: {sorted(valores)}")
    
    if set(valores).issubset({0, 1}):
        print("  • ✅ Es una variable binaria válida (0/1)")
    else:
        print("  • ❌ La variable NO es binaria (0/1)")

# ============================================================
# 9. VERIFICAR RELACIONES CLAVE
# ============================================================

print("\n" + "="*70)
print("8. RELACIONES CLAVE")
print("="*70)

# Tabla de contingencia: Informalidad vs Crédito formal 2024
if 'trabajador_informal' in df.columns and 'credito_formal_2024' in df.columns:
    print("\n📊 Tabla: trabajador_informal × credito_formal_2024")
    print("-"*70)
    tabla = pd.crosstab(df['trabajador_informal'], df['credito_formal_2024'])
    print(tabla)
    print("\nInterpretación:")
    print(f"  • Informales sin crédito formal 2024: {tabla.get(1, {}).get(0, 0):,}")
    print(f"  • Informales con crédito formal 2024: {tabla.get(1, {}).get(1, 0):,}")

# Tabla de contingencia: Billetera vs Crédito informal (solo informales)
if 'trabajador_informal' in df.columns:
    df_informal = df[df['trabajador_informal'] == 1]
    
    print("\n📊 Tabla: usa_billetera × credito_informal_2025 (solo INFORMALES)")
    print("-"*70)
    tabla_billetera = pd.crosstab(df_informal['usa_billetera'], df_informal['credito_informal_2025'])
    print(tabla_billetera)
    
    # Porcentajes
    print("\n📊 Porcentajes por grupo de billetera:")
    for billetera in [0, 1]:
        sub = df_informal[df_informal['usa_billetera'] == billetera]
        if len(sub) > 0:
            pct_informal = sub['credito_informal_2025'].mean()
            print(f"  • {'Sin' if billetera == 0 else 'Con'} billetera (N={len(sub):,}): {pct_informal:.2%} tiene crédito informal")

# ============================================================
# 10. CORRELACIONES BÁSICAS
# ============================================================

print("\n" + "="*70)
print("9. CORRELACIONES BÁSICAS")
print("="*70)

# Seleccionar variables clave para correlación
variables_correlacion = ['credito_informal_2025', 'usa_billetera', 
                         'trabajador_informal', 'credito_formal_2024',
                         'nuevo_credito_formal', 'tiene_barrera_credito',
                         'edad', 'nivel_educativo', 'log_ingreso_2024']

# Crear log_ingreso si no existe
if 'ingreso_percapita_2024' in df.columns:
    df['log_ingreso_2024'] = np.log(df['ingreso_percapita_2024'])
    variables_correlacion.append('log_ingreso_2024')

# Filtrar variables que existen
variables_existentes = [v for v in variables_correlacion if v in df.columns]

print(f"\n📊 Matriz de correlación (variables seleccionadas):")
print("-"*70)
corr_matrix = df[variables_existentes].corr()
print(corr_matrix.round(4))

# ============================================================
# 11. VERIFICACIÓN FINAL
# ============================================================

print("\n" + "="*70)
print("10. RESUMEN DE VERIFICACIÓN")
print("="*70)

# Contar problemas
problemas = []

# 1. Variables con tipo incorrecto
for col in df.columns:
    tipo_actual = str(df[col].dtype)
    tipo_esperado = tipos_esperados.get(col)
    if tipo_esperado:
        if 'int' in tipo_actual and 'int' in tipo_esperado:
            pass
        elif 'float' in tipo_actual and ('float' in tipo_esperado or 'int' in tipo_esperado):
            pass
        elif tipo_actual != tipo_esperado:
            problemas.append(f"Tipo incorrecto en '{col}': {tipo_actual} (esperado: {tipo_esperado})")

# 2. Variables con missing
for col in df.columns:
    if df[col].isna().sum() > 0:
        problemas.append(f"Valores missing en '{col}': {df[col].isna().sum():,}")

# 3. Variables binarias con valores extraños
for col in ['jefe_hogar', 'ocupado', 'trabajador_informal', 'tiene_billetera', 
            'usa_billetera', 'credito_formal_2024', 'credito_formal_2025',
            'credito_informal_2025', 'nuevo_credito_formal']:
    if col in df.columns:
        valores = df[col].dropna().unique()
        if not set(valores).issubset({0, 1}):
            problemas.append(f"Variable '{col}' tiene valores extraños: {sorted(valores)}")

# Reportar
if problemas:
    print("\n⚠️ PROBLEMAS ENCONTRADOS:")
    for p in problemas:
        print(f"  • {p}")
else:
    print("\n✅ TODAS LAS VARIABLES ESTÁN CORRECTAS")
    print("   • Tipos de datos: OK")
    print("   • Valores missing: OK")
    print("   • Rangos: OK")
    print("   • Variables binarias: OK")

print("\n" + "="*70)
print("✅ VERIFICACIÓN COMPLETADA")
print("="*70)

VERIFICACIÓN COMPLETA DE BASE_H3.csv

✅ Base cargada: 7,708 filas, 31 columnas

1. ESTRUCTURA GENERAL

📊 Dimensiones: 7,708 filas x 31 columnas
💾 Memoria usada: 2.22 MB

2. TIPOS DE DATOS POR VARIABLE

📋 Tipos de datos:
----------------------------------------------------------------------
Variable                                Tipo actual   Tipo esperado       ¿OK?
----------------------------------------------------------------------
id_persona                                   object          object          ✅
jefe_hogar                                    int64           int64          ✅
sexo                                          int64          object          ❌
edad                                        float64           int64          ✅
ocupado                                       int64           int64          ✅
trabajador_informal                         float64           int64          ✅
tiene_billetera                               int64           int64          ✅
usa_bi